# WIA3003: Academic Project II
## QCNN Baseline Model


## 1.0 Initial Model Design

The FYP I report, Section 5.7, proposes 4-8 qubits, angle encoding in [0, pi], one or two convolution layers and one pooling stage. The overview specifies PennyLane. This initial implementation uses the eight existing measurements, not CT images or raw coordinates.

| Component | Initial choice |
| --- | --- |
| Inputs | Eight measurements in the saved feature order |
| Qubits | Eight, one per input |
| Encoding | RY rotations after training-only min-max scaling |
| Convolution | Two layers of neighbouring two-qubit blocks |
| Parameter sharing | One set of six angles per convolution layer |
| Pooling | Controlled RY on each even/odd pair; retain odd wires |
| Readout | Mean of four retained-wire Pauli-Z expectations |
| Probability | p(male) = (1 - mean Z) / 2 |
| Simulator | default.qubit, analytic expectations, no finite shots |

This is a shallow QCNN-inspired adaptation for tabular data. It is not an exact replication of Cong et al. Neighbouring-feature order and pooling/readout choices need justification and validation. It does not imply a quantum speed or accuracy advantage.


## 2.0 Required Packages

Install missing packages in the selected notebook environment, then restart the kernel.

```python
%pip install pennylane pandas scikit-learn matplotlib xlrd joblib
```


In [ ]:
import pennylane as qml
import numpy as np

print("PennyLane version:", qml.__version__)


## 3.0 Circuit Preparation

### 3.1 Convolution and Pooling Blocks

Each convolution block has two trainable single-qubit rotations and a CNOT. Parameters are shared across positions within each layer. Pooling applies a controlled rotation before retiring the source wire from later operations and readout. The simulator still stores all eight wires. Ignoring retired wires at readout gives reduced-state statistics; it does not physically remove qubits.

The gates below are project design choices. They are not copied from the reference tutorial.


In [ ]:
n_qubits = 8
surviving_wires = [1, 3, 5, 7]

def convolution_block(parameters, first, second):
    qml.Rot(*parameters[:3], wires=first)
    qml.Rot(*parameters[3:], wires=second)
    qml.CNOT(wires=[first, second])

def convolution_layer(parameters):
    # Apply the same block across neighbouring pairs.
    for first in [0, 2, 4, 6]:
        convolution_block(parameters, first, first + 1)
    for first in [1, 3, 5]:
        convolution_block(parameters, first, first + 1)

def pooling_layer(angle):
    for source, target in [(0, 1), (2, 3), (4, 5), (6, 7)]:
        qml.CRY(angle, wires=[source, target])

device = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(device, interface="autograd", diff_method="backprop")
def qcnn_circuit(inputs, weights):
    for wire in range(n_qubits):
        qml.RY(inputs[..., wire], wires=wire)
    convolution_layer(weights[:6])
    convolution_layer(weights[6:12])
    pooling_layer(weights[12])
    return [qml.expval(qml.PauliZ(wire)) for wire in surviving_wires]



### 3.2 Circuit Checking

This checks valid probabilities, finite nonzero gradients and the parameter count. These are untrained outputs, not research results. No test data are used.


In [ ]:
from pennylane import numpy as pnp

rng = np.random.default_rng(42)
weights = pnp.array(rng.uniform(-0.1, 0.1, 13), requires_grad=True)
example_angles = pnp.array(np.linspace(0.2, 2.8, n_qubits), requires_grad=False)

def probability_male(parameters):
    expectations = pnp.stack(qcnn_circuit(example_angles, parameters))
    return (1 - pnp.mean(expectations)) / 2

probability = probability_male(weights)
gradient = qml.grad(probability_male)(weights)
assert np.isfinite(float(probability)) and 0 <= float(probability) <= 1
assert np.isfinite(gradient).all()
assert np.linalg.norm(gradient) > 1e-10
print("Trainable parameters:", len(weights))
print("Untrained example probability:", float(probability))
print("Gradient norm:", float(np.linalg.norm(gradient)))
print(qml.draw(qcnn_circuit)(example_angles, weights))



## 4.0 Dataset Preparation
### 4.1 Load Dataset and Saved Splits
Use the same current workbook, outer split and inner validation split as the CNN experiment. Stop if the workbook has changed again.


In [ ]:
import json, hashlib, time, platform
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay)

current_folder = Path.cwd()
project_folder = current_folder.parent if current_folder.name == 'notebooks' else current_folder
assert (project_folder / 'notebooks').is_dir()
output_folder = project_folder / 'results' / 'qcnn_baseline'
output_folder.mkdir(parents=True, exist_ok=True)
with open(project_folder / 'results' / 'cnn_baseline' / 'run_details.json', encoding='utf-8') as file:
    split_details = json.load(file)
data_path = Path.home() / 'Downloads' / 'Orbital, Nasal and Craniofacial Class Pattern.xls'
dataset_hash = hashlib.sha256(data_path.read_bytes()).hexdigest()
assert dataset_hash == split_details['dataset_sha256'], 'Workbook changed since CNN run. Review data first.'
df = pd.read_excel(data_path, sheet_name='Raw Data')
df.columns = df.columns.str.replace(r'\s+', ' ', regex=True).str.strip()
features = split_details['features']
X = df[features].apply(pd.to_numeric, errors='raise')
y = df['Gender'].astype('string').str.strip().map(split_details['label_mapping'])
assert np.isfinite(X.to_numpy()).all() and not y.isna().any()
y = y.astype(int)
train_indices = np.array(split_details['train_row_positions'])
test_indices = np.array(split_details['test_row_positions'])
fit_indices = np.array(split_details['fit_row_positions'])
val_indices = np.array(split_details['validation_row_positions'])
assert np.array_equal(np.sort(np.concatenate([train_indices, test_indices])), np.arange(len(X)))
assert np.array_equal(np.sort(np.concatenate([fit_indices, val_indices])), np.sort(train_indices))
groups = X.groupby(features, sort=False, dropna=False).ngroup()
assert not (set(groups.iloc[train_indices]) & set(groups.iloc[test_indices]))
assert not (set(groups.iloc[fit_indices]) & set(groups.iloc[val_indices]))
print('Training:', X.iloc[train_indices].shape)
print('Testing:', X.iloc[test_indices].shape)
print('Fitting:', X.iloc[fit_indices].shape)
print('Validation:', X.iloc[val_indices].shape)


### 4.2 Angle Scaling
Fit min-max scaling on the fitting subset only. Clip validation values outside its range to [0, pi], and report the clipping count.


In [ ]:
selection_scaler = MinMaxScaler(feature_range=(0, np.pi), clip=True)
X_fit = pnp.array(selection_scaler.fit_transform(X.iloc[fit_indices]), requires_grad=False)
X_val = pnp.array(selection_scaler.transform(X.iloc[val_indices]), requires_grad=False)
y_fit = pnp.array(y.iloc[fit_indices].to_numpy(), requires_grad=False)
y_val = pnp.array(y.iloc[val_indices].to_numpy(), requires_grad=False)
validation_clipped = int(((X.iloc[val_indices].to_numpy() < selection_scaler.data_min_) |
                          (X.iloc[val_indices].to_numpy() > selection_scaler.data_max_)).sum())
print('Validation feature values clipped:', validation_clipped)


## 5.0 QCNN Training
### 5.1 Loss Function and Training Settings
Use binary cross-entropy and Adam. The quantum readout estimates male probability directly, without a classical trainable head. Batch broadcasting evaluates several records together. Settings are fixed before looking at test results.

Optimizer reference: https://docs.pennylane.ai/en/stable/code/api/pennylane.AdamOptimizer.html


In [ ]:
SEED = 42
MAX_EPOCHS = 30
PATIENCE = 5
BATCH_SIZE = 32
LEARNING_RATE = 0.02

# Works with a single record or a batch of records.
def predict_probability(parameters, inputs):
    expectations = pnp.stack(qcnn_circuit(inputs, parameters))
    return (1 - pnp.mean(expectations, axis=0)) / 2

def loss_function(parameters, inputs, labels):
    probabilities = pnp.clip(predict_probability(parameters, inputs), 1e-7, 1 - 1e-7)
    return -pnp.mean(labels * pnp.log(probabilities) + (1 - labels) * pnp.log(1 - probabilities))

def initialise_weights():
    return pnp.array(np.random.default_rng(SEED).uniform(-0.1, 0.1, 13), requires_grad=True)

def run_epoch(parameters, optimiser, inputs, labels, order):
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start:start + BATCH_SIZE]
        parameters = optimiser.step(lambda current: loss_function(current, inputs[batch], labels[batch]), parameters)
    return parameters


### 5.2 Small Batch Checking
Check that broadcasting matches single-record predictions and that a finite gradient update completes. This is not a model-performance result.


In [ ]:
check_weights = initialise_weights()
start_time = time.perf_counter()
batched = np.asarray(predict_probability(check_weights, X_fit[:4]))
single = np.array([float(predict_probability(check_weights, row)) for row in X_fit[:4]])
np.testing.assert_allclose(batched, single, rtol=1e-6, atol=1e-8)
check_loss = lambda current: loss_function(current, X_fit[:4], y_fit[:4])
gradient = qml.grad(check_loss)(check_weights)
assert np.isfinite(gradient).all() and np.linalg.norm(gradient) > 1e-10
updated_weights = qml.AdamOptimizer(stepsize=LEARNING_RATE).step(check_loss, check_weights)
assert np.isfinite(float(check_loss(updated_weights)))
print('Initial batch loss:', float(check_loss(check_weights)))
print('Loss after one update:', float(check_loss(updated_weights)))
print('Batch check seconds:', round(time.perf_counter() - start_time, 2))


### 5.3 Model Training and Validation
Select the epoch with the lowest validation loss. Stop after five epochs without improvement, or at the 30-epoch limit. Preserve the history and selected weights for review.


In [ ]:
selection_weights = initialise_weights()
optimiser = qml.AdamOptimizer(stepsize=LEARNING_RATE)
training_rng = np.random.default_rng(SEED)
best_loss = np.inf
best_epoch = 0
waiting = 0
history_rows = []
selection_started = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    selection_weights = run_epoch(selection_weights, optimiser, X_fit, y_fit,
                                  training_rng.permutation(len(X_fit)))
    train_loss = float(loss_function(selection_weights, X_fit, y_fit))
    val_loss = float(loss_function(selection_weights, X_val, y_val))
    assert np.isfinite(train_loss) and np.isfinite(val_loss)
    history_rows.append({'Epoch': epoch, 'Training loss': train_loss, 'Validation loss': val_loss})
    pd.DataFrame(history_rows).to_csv(output_folder / 'training_history.csv', index=False)
    print(f'Epoch {epoch}: training loss={train_loss:.4f}, validation loss={val_loss:.4f}', flush=True)
    if val_loss < best_loss:
        best_loss, best_epoch, waiting = val_loss, epoch, 0
        np.save(output_folder / 'validation_best_weights.npy', np.asarray(selection_weights))
    else:
        waiting += 1
    if waiting >= PATIENCE:
        break
selection_seconds = time.perf_counter() - selection_started
history = pd.DataFrame(history_rows)
print('Selected epochs:', best_epoch)
history.plot(x='Epoch', y=['Training loss', 'Validation loss'], figsize=(7, 4))
plt.ylabel('Binary cross-entropy loss')
plt.tight_layout()
plt.savefig(output_folder / 'training_history.png', dpi=200)
plt.show()


### 5.4 Final Model Training
Fit a fresh scaler on all outer training records and initialise fresh circuit weights. Train for the selected number of epochs. Test data do not determine the training duration.


In [ ]:
scaler = MinMaxScaler(feature_range=(0, np.pi), clip=True)
X_train = pnp.array(scaler.fit_transform(X.iloc[train_indices]), requires_grad=False)
X_test = pnp.array(scaler.transform(X.iloc[test_indices]), requires_grad=False)
y_train = pnp.array(y.iloc[train_indices].to_numpy(), requires_grad=False)
y_test = y.iloc[test_indices].to_numpy()
test_clipped = int(((X.iloc[test_indices].to_numpy() < scaler.data_min_) |
                    (X.iloc[test_indices].to_numpy() > scaler.data_max_)).sum())
final_weights = initialise_weights()
optimiser = qml.AdamOptimizer(stepsize=LEARNING_RATE)
training_rng = np.random.default_rng(SEED)
refit_started = time.perf_counter()
for epoch in range(1, best_epoch + 1):
    final_weights = run_epoch(final_weights, optimiser, X_train, y_train,
                              training_rng.permutation(len(X_train)))
    print(f'Final training epoch {epoch}/{best_epoch}', flush=True)
refit_seconds = time.perf_counter() - refit_started
print('Final training completed.')


## 6.0 Model Evaluation
The fixed threshold is 0.5. Male is the positive class. These are analytic noiseless simulator results, not quantum-hardware results.


In [ ]:
probability_male = np.asarray(predict_probability(final_weights, X_test), dtype=float)
assert np.isfinite(probability_male).all()
predictions = (probability_male >= 0.5).astype(int)
metrics = {
    'Accuracy': accuracy_score(y_test, predictions),
    'Precision (Male)': precision_score(y_test, predictions, zero_division=0),
    'Recall (Male)': recall_score(y_test, predictions, zero_division=0),
    'F1 (Male)': f1_score(y_test, predictions, zero_division=0),
    'Macro F1': f1_score(y_test, predictions, average='macro', zero_division=0),
    'ROC AUC': roc_auc_score(y_test, probability_male),
}
previous_results = pd.read_csv(project_folder / 'results' / 'cnn_baseline' / 'model_comparison.csv', index_col='Model')
comparison = pd.concat([previous_results, pd.DataFrame([metrics], index=pd.Index(['QCNN'], name='Model'))])
display(comparison.round(4))
print(classification_report(y_test, predictions, labels=[0, 1], target_names=['Female', 'Male'], digits=4, zero_division=0))
cm = confusion_matrix(y_test, predictions, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=['Female', 'Male']).plot(cmap='Blues', values_format='d', colorbar=False)
plt.title('QCNN initial baseline')
plt.tight_layout()
plt.savefig(output_folder / 'confusion_matrix.png', dpi=200)
plt.show()


## 7.0 Save Model and Results
The circuit code, feature order, scaler and weights are all needed for reloading. No original dataset identifiers are exported.


In [ ]:
np.save(output_folder / 'qcnn_weights.npy', np.asarray(final_weights))
joblib.dump(scaler, output_folder / 'scaler.joblib')
comparison.to_csv(output_folder / 'model_comparison.csv')
pd.DataFrame([metrics]).to_csv(output_folder / 'metrics.csv', index=False)
pd.DataFrame(cm, index=['Actual Female', 'Actual Male'], columns=['Predicted Female', 'Predicted Male']).to_csv(output_folder / 'confusion_matrix.csv')
pd.DataFrame({'source_row_position': test_indices, 'actual_label': y_test,
    'predicted_label': predictions, 'probability_male': probability_male}).to_csv(output_folder / 'test_predictions.csv', index=False)
run_details = {
    'dataset_sha256': dataset_hash, 'features': features,
    'label_mapping': split_details['label_mapping'],
    'train_row_positions': train_indices.tolist(), 'test_row_positions': test_indices.tolist(),
    'fit_row_positions': fit_indices.tolist(), 'validation_row_positions': val_indices.tolist(),
    'architecture': 'Two shared brickwork Rot-Rot-CNOT layers, CRY pooling, mean Z on odd wires',
    'qubits': 8, 'parameters': 13, 'shots': None, 'device': 'default.qubit',
    'gradient_method': 'backprop', 'seed': SEED, 'learning_rate': LEARNING_RATE,
    'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'selected_epochs': best_epoch,
    'batch_size': BATCH_SIZE, 'threshold': 0.5,
    'validation_clipped_feature_values': validation_clipped,
    'test_clipped_feature_values': test_clipped,
    'selection_seconds': selection_seconds, 'refit_seconds': refit_seconds,
    'pennylane_version': qml.__version__, 'python_version': platform.python_version(),
    'metrics': {key: float(value) for key, value in metrics.items()},
}
with open(output_folder / 'run_details.json', 'w', encoding='utf-8') as file:
    json.dump(run_details, file, indent=2)
print('Saved to:', output_folder)


### 7.1 Saved Model Checking


In [ ]:
loaded_weights = pnp.array(np.load(output_folder / 'qcnn_weights.npy', allow_pickle=False), requires_grad=False)
loaded_scaler = joblib.load(output_folder / 'scaler.joblib')
loaded_inputs = pnp.array(loaded_scaler.transform(X.iloc[test_indices]), requires_grad=False)
loaded_probabilities = np.asarray(predict_probability(loaded_weights, loaded_inputs))
np.testing.assert_allclose(probability_male, loaded_probabilities, rtol=1e-6, atol=1e-8)
assert np.array_equal(predictions, (loaded_probabilities >= 0.5).astype(int))
print('Saved weights and scaler loaded successfully; predictions match.')


## 8.0 Results and Discussion
Write the interpretation after training. Compare class-specific errors and macro F1, not accuracy alone. This is a shallow QCNN-inspired tabular adaptation on one split and seed, not evidence of quantum advantage. The fixed feature order and limited expressiveness need discussion. Investigate conflicting labels with the data provider without changing labels here. Exact feature groups do not guarantee subject independence. Do not use test performance to select new settings.


## 9.0 Readout Diagnosis

This section investigates the saved initial model without overwriting its weights, metrics or test predictions. Only inner fitting and validation records are used. The outer test set is not used to select changes.

### 9.1 Recheck Selected Model

Use the weights selected at epoch 13 and reconstruct its fitting-only scaler. The final refitted model uses a different scaler and must not be mixed with these weights.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

diagnostic_folder = output_folder / 'diagnostics'
diagnostic_folder.mkdir(parents=True, exist_ok=True)
with open(output_folder / 'run_details.json', encoding='utf-8') as file:
    diagnostic_run = json.load(file)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == diagnostic_run['dataset_sha256']
diag_fit_rows = np.array(diagnostic_run['fit_row_positions'])
diag_val_rows = np.array(diagnostic_run['validation_row_positions'])
diag_labels = df['Gender'].astype('string').str.strip().map(diagnostic_run['label_mapping']).to_numpy(dtype=int)
diag_scaler = MinMaxScaler(feature_range=(0, np.pi), clip=True)
diag_fit = pnp.array(diag_scaler.fit_transform(X.iloc[diag_fit_rows]), requires_grad=False)
diag_val = pnp.array(diag_scaler.transform(X.iloc[diag_val_rows]), requires_grad=False)
diag_weights = pnp.array(np.load(output_folder / 'validation_best_weights.npy', allow_pickle=False), requires_grad=True)

diagnostic_rows = []
for name, inputs, rows in [('Fitting', diag_fit, diag_fit_rows), ('Validation', diag_val, diag_val_rows)]:
    probabilities = np.asarray(predict_probability(diag_weights, inputs))
    diagnostic_rows.append({'Partition': name,
        'Accuracy': accuracy_score(diag_labels[rows], probabilities >= 0.5),
        'Loss': log_loss(diag_labels[rows], probabilities),
        'ROC AUC': roc_auc_score(diag_labels[rows], probabilities)})
diagnostic_metrics = pd.DataFrame(diagnostic_rows).set_index('Partition')
display(diagnostic_metrics.round(4))
diagnostic_metrics.to_csv(diagnostic_folder / 'selected_model_metrics.csv')



,Accuracy,Loss,ROC AUC
Partition,,,
Fitting,0.6883,0.6023,0.7424
Validation,0.6726,0.6073,0.7269


### 9.2 Parameter Sensitivity

Inspect the Jacobian on 32 fitting examples. Nonzero column norms mean each parameter changes some output locally. Full local rank is not proof of global identifiability or easy optimisation.


In [ ]:
probe = diag_fit[:32]
probe_output = lambda current: pnp.ravel(predict_probability(current, probe))
jacobian = np.asarray(qml.jacobian(probe_output)(diag_weights))
sensitivity = pd.DataFrame({
    'Parameter': np.arange(13),
    'Jacobian column norm': np.linalg.norm(jacobian, axis=0),
})
display(sensitivity)
print('Local Jacobian rank:', np.linalg.matrix_rank(jacobian, tol=1e-8))
sensitivity.to_csv(diagnostic_folder / 'parameter_sensitivity.csv', index=False)



Local Jacobian rank: 13


,Parameter,Jacobian column norm
0,0,0.057469
1,1,1.747121
2,2,0.076239
3,3,0.030101
4,4,2.556575
5,5,0.038761
6,6,0.028649
7,7,0.847933
8,8,0.005117
9,9,0.037218


### 9.3 Separate-Wire Readout

Freeze the selected quantum circuit and fit one logistic-regression readout to its four separate expectations. Use fitting records only to train the readout. This tests whether fixed equal averaging loses useful signal. It is a quantum-feature/classical-readout diagnostic, not an end-to-end retrained QCNN or a final model selection result.


In [ ]:
quantum_fit = np.asarray(qcnn_circuit(diag_fit, diag_weights)).T
quantum_val = np.asarray(qcnn_circuit(diag_val, diag_weights)).T
readout = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
readout.fit(quantum_fit, diag_labels[diag_fit_rows])
readout_probabilities = readout.predict_proba(quantum_val)[:, list(readout.classes_).index(1)]
fixed_probabilities = np.asarray(predict_probability(diag_weights, diag_val))
readout_rows = []
for name, probabilities in [('Fixed average', fixed_probabilities), ('Separate-wire logistic readout', readout_probabilities)]:
    readout_rows.append({'Readout': name,
        'Validation accuracy': accuracy_score(diag_labels[diag_val_rows], probabilities >= 0.5),
        'Validation loss': log_loss(diag_labels[diag_val_rows], probabilities),
        'Validation ROC AUC': roc_auc_score(diag_labels[diag_val_rows], probabilities)})
readout_comparison = pd.DataFrame(readout_rows).set_index('Readout')
display(readout_comparison.round(4))
readout_comparison.to_csv(diagnostic_folder / 'readout_validation_comparison.csv')
joblib.dump(readout, diagnostic_folder / 'frozen_feature_readout.joblib')
joblib.dump(diag_scaler, diagnostic_folder / 'selection_scaler.joblib')
np.save(diagnostic_folder / 'selection_quantum_weights.npy', np.asarray(diag_weights))
print('Saved diagnostic only. Original baseline results unchanged.')



Saved diagnostic only. Original baseline results unchanged.


,Validation accuracy,Validation loss,Validation ROC AUC
Readout,,,
Fixed average,0.6726,0.6073,0.7269
Separate-wire logistic readout,0.7500,0.5113,0.8192


### 9.4 Findings

The selected circuit achieved 68.83% fitting accuracy and 67.26% validation accuracy. Training and validation losses were close. This is consistent with limited fitting; it does not prove a unique cause. All 13 parameter entries had nonzero local sensitivity.

Keeping the four quantum outputs separate and fitting a logistic readout increased validation accuracy to 75.00%, reduced validation loss from 0.6073 to 0.5113, and increased validation ROC AUC from 0.7269 to 0.8192. This supports investigating the readout before adding qubits or circuit depth. It does not prove improvement on unseen data.

Next experiment: define an end-to-end hybrid quantum/classical readout using fitting/validation data, and use multiple training seeds. Declare it as a hybrid QCNN-inspired model. Preserve the original 69.05% hold-out result and avoid repeated test-guided model changes. The same validation set has now informed a design choice, so independent group-aware validation is needed for a stronger conclusion.


## 10.0 References and Notes

- FYP I report, Section 5.7: initial circuit scope and encoding.
- FYP I overview, slide 12: PennyLane implementation plan.
- Cong, Choi and Lukin (2019), Quantum Convolutional Neural Networks: https://arxiv.org/abs/1810.03787
- Qiskit Machine Learning QCNN tutorial, Sections 1.2 and 2: alternating convolution/pooling and inactive-wire interpretation. https://qiskit-community.github.io/qiskit-machine-learning/tutorials/11_quantum_convolutional_neural_networks.html

The original QCNN work studies quantum-state tasks. Applying these ideas to eight classical morphometric measurements is an adaptation. Gate choices, feature ordering and readout must be described explicitly.

### 10.1 AI Usage

Codex assisted with preparing this design and initial circuit code. Retain the assistance record and review the choices before implementation and reporting.
